# 04 · Protein in, ranked molecules out

The full inference pipeline:

```
protein PDB
   └─ detect binding sites        LIGSITE (builtin) / fpocket / P2Rank
        └─ pocket graph           whole residues within 8 Å, pocket-centred
             └─ diffusion sample  N molecules per pocket, size from the prior
                  └─ bond perception   RDKit DetermineBonds, geometric fallback
                       └─ score        affinity pK + QED + SA + clash/buriedness
                            └─ rank    CSV + SDF in the crystal frame
```

**Inputs** → notebook 02's output (generator), notebook 03's (ranker), notebook 01's
(graphs + library).

### On binding-site detectors
`method="auto"` tries **P2Rank**, then **fpocket**, then the builtin **LIGSITE**, and reports
which one ran in `Pocket.source`. The builtin needs no internet and no JVM — which matters,
because a Kaggle notebook that writes a dataset output usually has networking off. Measured on
30 held-out BioLiP complexes (a hit = the cavity covers ≥30 % of the true ligand's atoms
within 3 Å): **76.7 % top-1, 86.7 % top-3** at ~1 s per protein.

**DeepSite** is deliberately not wrapped: it is a hosted PlayMolecule web service with no
redistributable local model, so it cannot be a dependency of an offline pipeline. Feed its
predictions in as explicit centres if you want to use it.

In [ ]:
# --- locate the molgen package ---------------------------------------------
# On Kaggle, make `molgen` importable in one of these ways:
#   1. Add the "molgen-src" Dataset (upload this repo's `src/` folder), or
#   2. Add the output of notebook 01 (it ships a copy of `src/molgen`), or
#   3. Run the notebook from a clone of the repo.
import glob
import subprocess
import sys
from pathlib import Path


def find_molgen():
    patterns = [
        "/kaggle/input/*/src", "/kaggle/input/*", "/kaggle/input/*/*",
        "/kaggle/working/src", "./src", "../src", ".", "..",
    ]
    for pattern in patterns:
        for candidate in sorted(glob.glob(pattern)):
            if (Path(candidate) / "molgen" / "__init__.py").is_file():
                return str(Path(candidate).resolve())
    return None


MOLGEN_SRC = find_molgen()
if MOLGEN_SRC is None:
    raise FileNotFoundError(
        "Could not find the `molgen` package.\n"
        "Attach the molgen-src dataset (the repo's src/ folder, containing molgen/), "
        "or the output of notebook 01, then re-run this cell."
    )
if MOLGEN_SRC not in sys.path:
    sys.path.insert(0, MOLGEN_SRC)
print("molgen source:", MOLGEN_SRC)

In [ ]:
# --- dependencies -----------------------------------------------------------
# torch_geometric is required. We deliberately avoid torch_scatter/torch_cluster
# (they need wheels matched to the exact torch build and are the usual cause of
# Kaggle install failures) -- molgen uses scipy KD-trees and native torch ops.
# rdkit is optional: without it you lose SMILES/QED/SA but nothing else.
def ensure(module: str, package: str | None = None) -> bool:
    try:
        __import__(module)
        return True
    except ImportError:
        pkg = package or module
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    try:
        __import__(module)
        return True
    except ImportError:
        print(f"  !! {module} unavailable")
        return False


HAS_PYG = ensure("torch_geometric")
HAS_RDKIT = ensure("rdkit")
assert HAS_PYG, "torch_geometric is required"
print("rdkit:", HAS_RDKIT)

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import molgen
from molgen import (
    affinity, chem, dataset, diffusion, egnn, evaluate, graphs, pdbio, pockets, sources,
)
from molgen.train_utils import (
    EMA, History, TimeBudget, count_parameters, device_info, grad_norm,
    load_checkpoint, save_checkpoint, set_seed, warmup_cosine_lambda,
)

warnings.filterwarnings("ignore", category=UserWarning)
set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("molgen", molgen.__version__, "| device:", device)
print(device_info())

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

## 1 · Optional · install a stronger pocket finder

Skip this and the builtin detector is used. Both need internet enabled in the session.

In [ ]:
# fpocket (fast, ~2 min to install)
# !apt-get -qq update && apt-get -qq install -y fpocket && fpocket --version

# P2Rank (most accurate; needs a JRE)
# !apt-get -qq update && apt-get -qq install -y default-jre > /dev/null
# !wget -q https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz
# !tar -xzf p2rank_2.4.2.tar.gz -C /kaggle/working
# P2RANK = "/kaggle/working/p2rank_2.4.2/prank"

import shutil
P2RANK = globals().get("P2RANK")
print("fpocket:", shutil.which("fpocket") or "not installed")
print("p2rank :", P2RANK or "not installed")
POCKET_METHOD = "auto"     # "auto" | "builtin" | "fpocket" | "p2rank"

## 2 · Load the trained models

In [ ]:
def locate(filename, extra=()):
    patterns = [
        f"/kaggle/input/*/{filename}", f"/kaggle/input/*/*/{filename}",
        f"/kaggle/input/**/{filename}", str(WORK / "**" / filename),
        str(WORK / filename), *extra,
    ]
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    return None


diff_path = locate("diffusion_best.pt") or locate("diffusion_final.pt") or locate("diffusion_last.pt")
aff_path = locate("affinity_best.pt") or locate("affinity_last.pt")
assert diff_path, "no diffusion checkpoint found — attach notebook 02's output"
print("generator:", diff_path)
print("ranker   :", aff_path or "NONE (molecules will be ranked without predicted affinity)")

dck = load_checkpoint(diff_path, map_location="cpu")
gen = diffusion.PocketLigandDiffusion(diffusion.DiffusionConfig(**dck["config"]))
# EMA weights generate visibly cleaner geometry than the raw ones at the same step.
if "ema" in dck:
    state = {k: v for k, v in dck["ema"]["shadow"].items()}
    gen.load_state_dict({**dck["model"], **state}, strict=False)
    print("loaded EMA weights")
else:
    gen.load_state_dict(dck["model"])
gen = gen.to(device).eval()

ranker = None
if aff_path:
    ack = load_checkpoint(aff_path, map_location="cpu")
    ranker = affinity.AffinityNet(affinity.AffinityConfig(**ack["config"]))
    ranker.load_state_dict(ack["model"])
    ranker = ranker.to(device).eval()

build = dck.get("extra", {}).get("meta", {}).get("build_config", {})
POCKET_RADIUS = build.get("pocket_radius", 8.0)
MAX_POCKET_ATOMS = build.get("max_pocket_atoms", 600)
POCKET_CUTOFF = build.get("pocket_cutoff", 4.5)
print(f"\ngenerator trained with pocket_radius={POCKET_RADIUS} max_pocket_atoms={MAX_POCKET_ATOMS}")
print(f"coord_scale={gen.cfg.coord_scale:.3f}  timesteps={gen.cfg.timesteps}")

prior_path = locate("size_prior.json", extra=["/kaggle/input/*/graphs/size_prior.json"])
size_prior = dataset.SizePrior.load(prior_path) if prior_path else dataset.SizePrior()
print("size prior:", prior_path or "default")

## 3 · Pick a target protein

Either point at a local/attached PDB, or fetch one from RCSB (needs internet).

In [ ]:
TARGET = "1b7z"          # RCSB id, or set TARGET_PDB to a path
TARGET_PDB = None

if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"/kaggle/input/**/{TARGET}*.pdb", recursive=True))
    if candidates:
        TARGET_PDB = candidates[0]
    else:
        import urllib.request
        TARGET_PDB = str(WORK / f"{TARGET}.pdb")
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{TARGET.upper()}.pdb", TARGET_PDB)
        print("downloaded from RCSB")

TARGET_PDB = str(TARGET_PDB)
receptor = pdbio.parse_pdb(TARGET_PDB)
print(f"{TARGET_PDB}\n{len(receptor):,} heavy atoms | "
      f"{len(np.unique(receptor.residue_group_ids())):,} residues | "
      f"chains {sorted(set(receptor.chain))}")

# A crystallised ligand, when present, gives a free check on the detector.
het = receptor[receptor.is_hetatm]
known_ligand = None
for resname in pd.Series(het.resname).value_counts().index:
    sub = het[het.resname == resname]
    keep, _ = chem.is_interesting_ligand(resname, len(sub), sub.element, 6, 64)
    if keep:
        known_ligand = sub
        print(f"reference ligand in the file: {resname} ({len(sub)} atoms)")
        break

## 4 · Detect binding sites

In [ ]:
t0 = time.time()
found = pockets.detect_pockets(
    TARGET_PDB, method=POCKET_METHOD, receptor=receptor, max_pockets=8,
    **({"prank": P2RANK} if POCKET_METHOD == "p2rank" and P2RANK else {}),
)
print(f"{len(found)} pockets in {time.time() - t0:.1f}s via '{found[0].source if found else '-'}'\n")
display(pd.DataFrame([p.summary() for p in found]))

if known_ligand is not None:
    det = pockets.score_detection(found, known_ligand.pos.mean(0), ligand_pos=known_ligand.pos)
    print("\ncheck against the crystal ligand:")
    print(f"  top-1 covers the ligand : {det['overlap_top1']}")
    print(f"  top-3 covers the ligand : {det['overlap_top3']}  (first hit at rank {det['overlap_rank']})")
    print(f"  centre distance, best of top 3: {det['dcc']:.2f} A")

## 5 · Build pocket graphs

The detector's **point cloud** is handed to the encoder, not just the centre — an elongated
cleft then selects the residues that actually line it. The encoding is identical to training,
which is what keeps the model in distribution.

In [ ]:
TOP_POCKETS = 3

pocket_samples = []
for p in found[:TOP_POCKETS]:
    ref = p.points if p.points is not None else p.center.reshape(1, 3)
    sample = graphs.encode_pocket_only(
        receptor, reference_pos=ref, complex_id=f"{TARGET}_p{p.rank}",
        pocket_radius=POCKET_RADIUS, max_pocket_atoms=MAX_POCKET_ATOMS,
    )
    if sample is None:
        print(f"pocket {p.rank}: too few atoms, skipped")
        continue
    pocket_samples.append((p, sample))
    print(f"pocket {p.rank}: {len(sample['poc_pos']):>4} atoms, "
          f"{sample['n_poc_res']:>3} residues, "
          f"size prior -> {[size_prior.sample(sample['n_poc_res']) for _ in range(5)]}")

assert pocket_samples, "no usable pocket"

## 6 · Generate

`eta=1.0` is ancestral DDPM sampling; `eta=0.0` is deterministic DDIM, which is sharper but
loses diversity — and diversity is the point when you are generating a candidate library.
`num_steps` below `cfg.timesteps` strides the schedule and trades quality for speed.

In [ ]:
N_PER_POCKET = 40
SAMPLE_STEPS = 300
ETA = 1.0
CHUNK = 10                 # molecules per forward batch

rng = np.random.default_rng(0)
all_records, provenance = [], []

for p, sample in pocket_samples:
    graph = graphs.featurize(sample, pocket_cutoff=POCKET_CUTOFF)
    poc_pos = graph.poc_pos.numpy()
    poc_elements = sample.get("poc_elements_str")
    made = 0
    t0 = time.time()

    while made < N_PER_POCKET:
        k = min(CHUNK, N_PER_POCKET - made)
        batch = diffusion.build_pocket_batch([graph] * k, device=device)
        n_atoms = [size_prior.sample(sample["n_poc_res"], rng) for _ in range(k)]
        out = gen.sample(batch, n_atoms=n_atoms, num_steps=SAMPLE_STEPS, eta=ETA)

        for mol in gen.to_molecules(out):
            rec = evaluate.evaluate_molecule(
                mol["elements"], mol["pos"], poc_pos=poc_pos, poc_elements=poc_elements,
                index=len(all_records), pocket_id=f"p{p.rank}",
            )
            all_records.append(rec)
            provenance.append((p, sample, mol))
        made += k

    print(f"pocket {p.rank}: {made} molecules in {time.time() - t0:.0f}s "
          f"({(time.time() - t0) / made:.1f}s each)", flush=True)

print(f"\n{len(all_records)} molecules generated")

## 7 · Score with the affinity model

Each generated molecule is inserted back into its pocket graph (`attach_ligand`) and scored.
Bond perception runs on the generated geometry, so a molecule that fails to sanitise still
gets geometry and clash numbers — it simply carries the invalid penalty into the ranking.

In [ ]:
if ranker is not None:
    complexes, keep_idx = [], []
    for i, (rec, (p, sample, mol)) in enumerate(zip(all_records, provenance)):
        c = graphs.attach_ligand(sample, mol["elements"], mol["pos"], pocket_cutoff=POCKET_CUTOFF)
        if c is not None:
            complexes.append(c)
            keep_idx.append(i)
    preds = affinity.predict_batched(ranker, complexes, device=str(device), batch_size=16).numpy()
    for i, pk in zip(keep_idx, preds):
        all_records[i].pred_affinity = float(pk)
    print(f"scored {len(preds)} molecules | predicted pK "
          f"mean {preds.mean():.2f} range [{preds.min():.2f}, {preds.max():.2f}]")
else:
    print("no ranker loaded — ranking on chemistry and geometry only")

## 8 · Rank

The composite score is affinity-dominated with QED and synthetic accessibility as
regularisers and hard geometry failures as penalties. Maximising predicted affinity alone
drifts towards large greasy molecules that the affinity model happens to like — which is the
standard failure mode of this kind of pipeline, not a hypothetical one. Adjust `RankWeights`
to match what you are actually screening for.

In [ ]:
weights = evaluate.RankWeights(
    affinity=1.0, qed=0.4, sa=0.3, buriedness=0.3,
    clash_penalty=0.5, fragment_penalty=1.0, invalid_penalty=3.0,
)
ranked = evaluate.rank_molecules(all_records, weights)

cols = [c for c in ["pocket_id", "score", "pred_affinity", "qed", "sa", "n_atoms",
                    "n_fragments", "n_clash", "buriedness", "mw", "logp", "smiles"]
        if c in ranked.columns]
display(ranked[cols].head(25))

print("\nset-level metrics:")
print(json.dumps(evaluate.set_metrics(all_records), indent=2))
print("\nper pocket:")
display(ranked.groupby("pocket_id").agg(
    n=("score", "size"), best_score=("score", "max"),
    mean_affinity=("pred_affinity", "mean"), validity=("valid", "mean"),
    clash_free=("n_clash", lambda s: (s == 0).mean()),
).round(3))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ok = ranked[ranked.valid] if "valid" in ranked else ranked
ax[0].hist(ranked.score, bins=30, color="#3b82f6"); ax[0].set_title("composite score")
if "pred_affinity" in ranked:
    ax[1].hist(ranked.pred_affinity.dropna(), bins=30, color="#10b981")
    ax[1].set_title("predicted pK")
if len(ok) and "qed" in ok:
    ax[2].scatter(ok.qed, ok.pred_affinity, s=12, alpha=0.5)
    ax[2].set_xlabel("QED"); ax[2].set_ylabel("predicted pK")
    ax[2].set_title("drug-likeness vs affinity")
ax[3].hist(ranked.n_atoms, bins=20, color="#8b5cf6"); ax[3].set_title("heavy atoms")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9 · Export

SDF poses are shifted back into the receptor's crystal frame, so they open directly next to
the original PDB in PyMOL or ChimeraX.

In [ ]:
ranked.to_csv(WORK / f"{TARGET}_ranked.csv", index=False)

by_index = {r.index: r for r in all_records}
prov_by_index = {r.index: pr for r, pr in zip(all_records, provenance)}

written = 0
for pid, group in ranked.groupby("pocket_id"):
    recs, offset = [], None
    for idx in group.head(20)["index"]:
        rec = by_index[int(idx)]
        recs.append(rec)
        offset = prov_by_index[int(idx)][1]["center"]
    if recs:
        written += evaluate.records_to_sdf(
            recs, WORK / f"{TARGET}_{pid}_top.sdf", offset=np.asarray(offset).reshape(3)
        )

print(f"wrote {TARGET}_ranked.csv and {written} poses across {ranked.pocket_id.nunique()} SDF files")
print("\nin PyMOL:")
print(f"  load {Path(TARGET_PDB).name}")
print(f"  load {TARGET}_p1_top.sdf")

## 10 · Optional · benchmark the detector on held-out complexes

Measures the *first* stage of the pipeline on the notebook-01 test split, where the true site
is known. Worth running once: if detection misses the site, nothing downstream can recover.

In [ ]:
BENCH_N = 25

try:
    biolip = sources.BioLiPSource()
    bench = biolip.specs(dedup=True, limit=3000)
    rng_b = np.random.default_rng(0)
    rows = []
    for spec in [bench[i] for i in rng_b.choice(len(bench), size=min(BENCH_N * 4, len(bench)), replace=False)]:
        rec = pdbio.parse_pdb(spec.receptor_path).heavy()
        lig = pdbio.parse_pdb(spec.ligand_path).heavy()
        keep, _ = chem.is_interesting_ligand(spec.ligand_name, len(lig), lig.element, 10, 50)
        if not keep or not (300 < len(rec) < 3000):
            continue
        pk = pockets.detect_pockets(str(spec.receptor_path), method=POCKET_METHOD,
                                    receptor=rec, max_pockets=10)
        s = pockets.score_detection(pk, lig.pos.mean(0), ligand_pos=lig.pos)
        rows.append({"id": spec.complex_id, "n_pockets": len(pk),
                     "top1": s["overlap_top1"], "top3": s["overlap_top3"], "dcc3": s["dcc"]})
        if len(rows) >= BENCH_N:
            break

    bdf = pd.DataFrame(rows)
    print(f"n={len(bdf)}  top-1 {bdf.top1.mean():.1%}  top-3 {bdf.top3.mean():.1%}  "
          f"median DCC(top3) {bdf.dcc3.median():.2f} A")
    display(bdf.head(15))
except FileNotFoundError:
    print("BioLiP source not attached — skipping the detector benchmark")

---
## Where to push this next

- **Train longer.** Validity is the metric that responds most to epochs. Resume notebook 02 in
  a fresh session via `RESUME`.
- **Scale the generator.** `hidden=256, num_layers=9` is the usual next step once the pipeline
  is known-good.
- **Close the loop.** `plan.md`'s "evaluate → improve → loop" is inpainting: keep the
  best-scoring fragment fixed and re-noise the rest. `sample()` already takes the pocket as
  frozen context, so fixing a ligand sub-graph is the same mechanism applied to ligand atoms.
- **Harder validation.** Re-dock the generated poses (smina/gnina) and compare the docking
  score against the ranker's prediction; a learned scorer that disagrees with docking on its
  own top picks is not ready to screen with.
- **Categorical types.** Element types use the EDM continuous relaxation here. A D3PM-style
  discrete diffusion on types usually improves validity.